In the land of Aethereia, where the sun dipped into the horizon and painted the sky with hues of crimson and gold, there was a village called Luminaria. Luminaria was known for its magical and mysterious properties, where the air was filled with a shimmering, ethereal light that seemed to emanate from the very essence of the land itself.

The village was inhabited by a diverse group of beings, each with their own unique abilities and strengths. There


LLm model is running locally by the help of llama.cpp

Lets load the PDF 

In [4]:
import pymupdf  # PyMuPDF

doc = pymupdf.open("handbook_qa_brief.pdf")
full_text = ""
for page in doc:
    full_text += page.get_text()


Lets create a chuncking of the pdf contents 

In [5]:
def chunk_text(text, chunk_size=300, overlap=50):
    """Split text into overlapping chunks."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append({
            "id": len(chunks),
            "text": chunk.strip(),
            "start": start,
            "end": min(end, len(text))
        })
        start += chunk_size - overlap  # slide forward with overlap
    return chunks

chunks = chunk_text(full_text, chunk_size=300, overlap=50)   

In [6]:
for c in chunks:
    print(f"Chunk {c['id']} (chars {c['start']}-{c['end']}):")
    print(c['text'][:100] + "...")
    print()

Chunk 0 (chars 0-300):
Handbook Q&A Assistant – Project brief
Page 1
Handbook Q&A Assistant
Project brief: plain-English an...

Chunk 1 (chars 250-550):
ts
them ask questions in plain English and get instant, cited answers drawn directly from the handbo...

Chunk 2 (chars 500-800):
s?”
“What is the notice period?”
Before and after
Today
With the assistant
Open a 30-page PDF and sk...

Chunk 3 (chars 750-1050):
inutes per lookup
Seconds per lookup
Output
•
A short, direct answer to the question.
•
A citation p...

Chunk 4 (chars 1000-1300):
than guessing.
Suggested approach
•
Ingest: extract text from the PDF or text file, keeping page and...

Chunk 5 (chars 1250-1550):
ges most relevant to the question.
•
Answer: have an LLM respond using only the retrieved passages, ...

Chunk 6 (chars 1500-1616):
wer includes a citation the employee can verify.
•
Out-of-scope questions are declined clearly inste...



Lets do embedding with the same model which is being infered by llama.cpp

In [8]:
from sentence_transformers import SentenceTransformer

# Load SmolLM2 as an embedding model
embed_model = SentenceTransformer("HuggingFaceTB/SmolLM2-135M")

# Embed the chunks (from previous chunking step)
embed_model.tokenizer.pad_token = embed_model.tokenizer.eos_token
# Embed the chunks
chunk_texts = [c["text"] for c in chunks]
embeddings = embed_model.encode(chunk_texts, show_progress_bar=True)
print(f"Number of chunks: {len(embeddings)}")
print(f"Embedding dimension: {embeddings.shape[1]}")
# Preview
for i, c in enumerate(chunks):
    print(f"\nChunk {i}: {c['text'][:80]}...")
    print(f"  Vector (first 5 dims): {embeddings[i][:5]}")

Batches: 100%|██████████| 1/1 [00:03<00:00,  3.33s/it]

Number of chunks: 7
Embedding dimension: 576

Chunk 0: Handbook Q&A Assistant – Project brief
Page 1
Handbook Q&A Assistant
Project bri...
  Vector (first 5 dims): [ 0.9765625   0.14453125 -2.828125   -0.7265625  -0.45703125]

Chunk 1: ts
them ask questions in plain English and get instant, cited answers drawn dire...
  Vector (first 5 dims): [ 0.52734375 -0.26953125  1.9296875   2.09375     0.15625   ]

Chunk 2: s?”
“What is the notice period?”
Before and after
Today
With the assistant
Open ...
  Vector (first 5 dims): [-2.953125   -1.3359375   1.3046875  -0.32617188  1.0078125 ]

Chunk 3: inutes per lookup
Seconds per lookup
Output
•
A short, direct answer to the ques...
  Vector (first 5 dims): [ 2.890625   -0.02478027  0.83984375  1.015625    1.515625  ]

Chunk 4: than guessing.
Suggested approach
•
Ingest: extract text from the PDF or text fi...
  Vector (first 5 dims): [-1.078125   -1.1796875  -0.08105469  0.734375    1.578125  ]

Chunk 5: ges most relevant to the question.
•
Ans

Lets implement vector search 

In [9]:
import numpy as np

def vector_search(query, embed_model, embeddings, chunks, top_k=2):
    """Search for the most relevant chunks given a query."""
    # Encode the query
    query_embedding = embed_model.encode([query])
    
    # Compute cosine similarity: dot product of normalized vectors
    # embeddings and query are already numpy arrays from .encode()
    scores = np.dot(embeddings, query_embedding.T).squeeze()
    norms = np.linalg.norm(embeddings, axis=1) * np.linalg.norm(query_embedding)
    cosine_scores = scores / norms
    
    # Get top-k indices
    top_indices = np.argsort(cosine_scores)[::-1][:top_k]
    
    results = []
    for idx in top_indices:
        results.append({
            "chunk_id": chunks[idx]["id"],
            "score": float(cosine_scores[idx]),
            "text": chunks[idx]["text"]
        })
    return results

# --- Try it out ---
queries = [
    "How many sick leaves do I get?",
    "What is the notice period?",
    "What is the suggested approach?"
]

for q in queries:
    print(f"🔍 Query: {q}")
    results = vector_search(q, embed_model, embeddings, chunks, top_k=2)
    for r in results:
        print(f"  ✅ [Score: {r['score']:.4f}] Chunk {r['chunk_id']}:")
        print(f"     {r['text'][:120]}...")
    print()


🔍 Query: How many sick leaves do I get?
  ✅ [Score: 0.8361] Chunk 5:
     ges most relevant to the question.
•
Answer: have an LLM respond using only the retrieved passages, returning citations ...
  ✅ [Score: 0.8317] Chunk 6:
     wer includes a citation the employee can verify.
•
Out-of-scope questions are declined clearly instead of invented....

🔍 Query: What is the notice period?
  ✅ [Score: 0.8933] Chunk 6:
     wer includes a citation the employee can verify.
•
Out-of-scope questions are declined clearly instead of invented....
  ✅ [Score: 0.8659] Chunk 5:
     ges most relevant to the question.
•
Answer: have an LLM respond using only the retrieved passages, returning citations ...

🔍 Query: What is the suggested approach?
  ✅ [Score: 0.8980] Chunk 6:
     wer includes a citation the employee can verify.
•
Out-of-scope questions are declined clearly instead of invented....
  ✅ [Score: 0.8853] Chunk 5:
     ges most relevant to the question.
•
Answer: have an LLM respond using o

Now let give this search result along with query to the LLM inference 

In [10]:
from openai import OpenAI

# Initialize client pointing to llama.cpp
client = OpenAI(
    base_url="http://127.0.0.1:9931/v1",
    api_key="sk-no-key-required"
)

def rag_answer(query, embed_model, embeddings, chunks, top_k=3):
    """Retrieve relevant chunks and generate an answer using SmolLM2."""
    
    # Step 1: Retrieve relevant chunks
    results = vector_search(query, embed_model, embeddings, chunks, top_k=top_k)
    
    # Step 2: Build context from retrieved chunks
    context = "\n\n".join([f"[Chunk {r['chunk_id']}]: {r['text']}" for r in results])
    
    # Step 3: Generate answer with LLM
    system_prompt = (
        "You are a helpful handbook Q&A assistant. "
        "Answer the user's question using ONLY the provided context. "
        "Include a citation referencing the chunk number. "
        "If the context doesn't contain the answer, say 'The handbook does not cover this topic.'"
    )
    
    user_prompt = f"Context:\n{context}\n\nQuestion: {query}"
    
    response = client.chat.completions.create(
        model="smollm2",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        max_tokens=200,
        temperature=0.3,
        top_p=0.95
    )
    
    answer = response.choices[0].message.content
    return answer, results

# --- Try it ---
query = "What is the suggested approach for building this assistant?"
answer, sources = rag_answer(query, embed_model, embeddings, chunks)

print(f"🔍 Query: {query}\n")
print(f"💬 Answer:\n{answer}\n")
print("📚 Sources:")
for s in sources:
    print(f"  Chunk {s['chunk_id']} (score: {s['score']:.4f}): {s['text'][:100]}...")


🔍 Query: What is the suggested approach for building this assistant?

💬 Answer:
[Chunk 2]: The suggested approach for building this assistant is to use a combination of natural language processing (NLP) and machine learning (ML) techniques to analyze and generate text based on the provided context. The assistant should be designed to:
•
Extract relevant information from the context and generate a response using only the retrieved passages.
•
Verify the accuracy of the response using a citation the employee can verify.
•
Use a citation the employee can verify to ensure the accuracy of the response.
•
Provide an answer that matches the sample questions above.
•
Provide an answer that includes a citation the employee can verify.
•
Use a citation the employee can verify to ensure the accuracy of the response.
•
Use a citation the employee can verify to ensure the accuracy of the response.
•
Use a citation the employee can verify to ensure the accuracy of the response.
•
Use a citation the 